# 01 — PCA & Clustering

**SDI Tehran Pipeline · Notebook 1 of 5**

## Purpose
Perform dimensionality reduction (PCA) and typological clustering (KMeans) on the full street network — formal Tehran plus the peri-urban periphery.

## Methodological note on PCA scope
PCA is fitted on the **training set only** (formal Tehran segments with surveyed attributes). The fitted scaler and PCA objects are then used to transform both the training set and the prediction set (periphery), ensuring no information from the prediction set leaks into the component space. This is the correct treatment for a supervised prediction pipeline.

KMeans clustering runs on the **full network** — this is for visualisation only (`whole_CH_2`) and does not feed into SDI prediction.

## Feature input
PCA and clustering both use **CH + INT only** (18 columns across 9 radii, non-normalised). These two families capture the structural hierarchy of the street network — movement potential and integration — without conflating it with NC or TD.

## Inputs
- `segments.geojson` — full network in WGS84
- `training_set.csv` — formal Tehran segments with surveyed attributes

## Outputs
- `data/outputs/segments_pca_clustered.csv` — segment ID, PC1, PC2, PC3, whole\_CH\_2
- `data/outputs/pca_variance.json` — explained variance ratios (paper reporting)
- `data/training_set.csv` — updated with PC1, PC2, PC3 columns appended
- `models/pca_scaler.joblib` — StandardScaler fitted on training set CH+INT
- `models/pca_model.joblib` — PCA(3) fitted on training set CH+INT

## 0 · Config & imports

In [1]:
import sys, json
import numpy as np
import pandas as pd
import geopandas as gpd
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
import joblib

sys.path.insert(0, '..')
from config import (
    SEGMENTS_GEOJSON, SEGMENTS_PCA_CLUSTERED_CSV, PCA_VARIANCE_JSON,
    TRAINING_SET_CSV, MODELS_DIR,
    PCA_INPUT_COLS, CLUSTERING_FEATURES,
    PCA_N_COMPONENTS, N_CLUSTERS, RANDOM_STATE,
    SEG_ID_COL,
    PCA_SCALER_PATH, PCA_MODEL_PATH
)

# Ensure models directory exists
Path(MODELS_DIR).mkdir(parents=True, exist_ok=True)

print("Config loaded.")
print(f"  PCA input cols : {len(PCA_INPUT_COLS)} (CH + INT across 9 radii)")
print(f"  Cluster count  : {N_CLUSTERS}")
print(f"  PCA components : {PCA_N_COMPONENTS}")

Config loaded.
  PCA input cols : 16 (CH + INT across 9 radii)
  Cluster count  : 12
  PCA components : 3


## 1 · Load full network

Load the full network GeoJSON (formal Tehran + periphery). Geometry is preserved for the output join. This dataset is used for KMeans clustering only — PCA is fitted on the training set in Section 3.

In [2]:
gdf = gpd.read_file(SEGMENTS_GEOJSON)
print(f"Loaded {len(gdf):,} segments  |  CRS: {gdf.crs}")

assert SEG_ID_COL in gdf.columns, f"Column '{SEG_ID_COL}' not found. Available: {list(gdf.columns)}"

n_before = len(gdf)
gdf = gdf.dropna(subset=PCA_INPUT_COLS)
n_dropped = n_before - len(gdf)
print(f"Dropped {n_dropped:,} rows with nulls in PCA input columns ({n_dropped/n_before:.1%})")

Loaded 427,100 segments  |  CRS: EPSG:4326
Dropped 0 rows with nulls in PCA input columns (0.0%)


## 2 · Load training set

The training set contains formal Tehran segments with surveyed socioeconomic attributes. PCA will be fitted on this subset only.

In [3]:
training_set = pd.read_csv(TRAINING_SET_CSV)
print(f"Training set loaded: {len(training_set):,} segments")
print(f"Columns: {list(training_set.columns)}")

# Verify all PCA input columns are present
missing = [c for c in PCA_INPUT_COLS if c not in training_set.columns]
assert not missing, f"Missing PCA input columns in training set: {missing}"

n_before = len(training_set)
training_set = training_set.dropna(subset=PCA_INPUT_COLS)
print(f"Dropped {n_before - len(training_set):,} rows with nulls in PCA input columns")

Training set loaded: 147,518 segments
Columns: ['Depthmap_R', 'Angular_Co', 'Connectivi', 'Segment_Le', 'mean_area', 'perimeter', 'landuse', 'mean_pop', 'dilap_index', 'policy_era', 'CH_10k', 'CH_1200', 'CH_2000', 'CH_20k', 'CH_400', 'CH_40k', 'CH_5000', 'CH_800', 'INT_10k', 'INT_1200', 'INT_2000', 'INT_20k', 'INT_400', 'INT_40k', 'INT_5000', 'INT_800', 'NC_10k', 'NC_1200', 'NC_2000', 'NC_20k', 'NC_400', 'NC_40k', 'NC_5000', 'NC_800', 'TD_10k', 'TD_1200', 'TD_2000', 'TD_20k', 'TD_400', 'TD_40k', 'TD_5000', 'TD_800']
Dropped 104 rows with nulls in PCA input columns


## 3 · Fit PCA on training set

**Methodology note:** The StandardScaler and PCA(3) are fitted on the training set (formal Tehran) only. This is the correct data engineering approach — fitting on the prediction set would constitute data leakage. The fitted objects are saved to disk and loaded in NB03 to transform the periphery segments into the same component space.

Feature input: CH + INT measures only (18 columns, non-normalised). These capture structural network hierarchy independently of normalised accessibility scores.

In [4]:
X_train = training_set[PCA_INPUT_COLS].values

# Fit scaler on training set
pca_scaler = StandardScaler()
X_train_scaled = pca_scaler.fit_transform(X_train)

# Fit PCA on scaled training set
pca_model = PCA(n_components=PCA_N_COMPONENTS, random_state=RANDOM_STATE)
X_train_pca = pca_model.fit_transform(X_train_scaled)

variance_ratios = pca_model.explained_variance_ratio_.tolist()
cumulative = float(np.cumsum(variance_ratios)[-1])

print("Explained variance per component (training set):")
for i, v in enumerate(variance_ratios, 1):
    print(f"  PC{i}: {v:.4f} ({v*100:.1f}%)")
print(f"  Cumulative: {cumulative:.4f} ({cumulative*100:.1f}%)")

Explained variance per component (training set):
  PC1: 0.5331 (53.3%)
  PC2: 0.1749 (17.5%)
  PC3: 0.1367 (13.7%)
  Cumulative: 0.8447 (84.5%)


## 4 · Save scaler and PCA objects

Both objects must be loaded (not refit) in NB03 when transforming periphery segments into the same component space.

In [5]:
joblib.dump(pca_scaler, PCA_SCALER_PATH)
joblib.dump(pca_model, PCA_MODEL_PATH)
print(f"Saved: {PCA_SCALER_PATH}")
print(f"Saved: {PCA_MODEL_PATH}")

Saved: D:\SID\models\pca_scaler.joblib
Saved: D:\SID\models\pca_model.joblib


## 5 · Save variance report

In [6]:
variance_report = {
    "explained_variance_ratio": variance_ratios,
    "cumulative_variance": cumulative,
    "n_components": PCA_N_COMPONENTS,
    "n_features_input": len(PCA_INPUT_COLS),
    "feature_set": "CH + INT (non-normalised, 9 radii)",
    "fitted_on": "training set (formal Tehran)"
}
with open(PCA_VARIANCE_JSON, "w") as f:
    json.dump(variance_report, f, indent=2)
print(f"Saved: {PCA_VARIANCE_JSON}")

Saved: D:\SID\data\outputs\pca_variance.json


## 6 · Update training set with PC1, PC2, PC3

Append the three principal components to the training set and save. These replace the raw 18 CH+INT columns as the feature input for NB02 model training.

In [7]:
training_set['PC1'] = X_train_pca[:, 0]
training_set['PC2'] = X_train_pca[:, 1]
training_set['PC3'] = X_train_pca[:, 2]

training_set.to_csv(TRAINING_SET_CSV, index=False)
print(f"Updated training set saved: {TRAINING_SET_CSV}")
print(f"Shape: {training_set.shape}")
print(training_set[['PC1', 'PC2', 'PC3']].describe().round(4))

Updated training set saved: D:\SID\data\training_set.csv
Shape: (147414, 45)
               PC1          PC2          PC3
count  147414.0000  147414.0000  147414.0000
mean        0.0000       0.0000      -0.0000
std         2.9206       1.6727       1.4788
min        -3.3724      -3.8436     -22.9642
25%        -1.9860      -0.7587      -0.8870
50%        -0.6987       0.0035       0.0368
75%         1.2071       0.6136       0.8612
max        33.9128      54.2015      23.6559


## 7 · KMeans clustering on full network

**Methodology note:** Clustering uses CH + INT measures across all radii (18 features) on the full street network. k=12 was established in prior analysis. Labels are stored as `whole_CH_2`. This column is **not** an SDI component — it is a structural typological axis used for dashboard visualisation (cluster boundary layer) only.

In [8]:
X_cluster = gdf[CLUSTERING_FEATURES].values
scaler_cluster = StandardScaler()
X_cluster_scaled = scaler_cluster.fit_transform(X_cluster)

kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE, n_init='auto')
cluster_labels = kmeans.fit_predict(X_cluster_scaled)

print(f"KMeans converged in {kmeans.n_iter_} iterations")
unique, counts = np.unique(cluster_labels, return_counts=True)
print("Cluster sizes:")
for u, c in zip(unique, counts):
    print(f"  Cluster {u:>2}: {c:,} segments")

KMeans converged in 67 iterations
Cluster sizes:
  Cluster  0: 12,594 segments
  Cluster  1: 100,066 segments
  Cluster  2: 858 segments
  Cluster  3: 49,394 segments
  Cluster  4: 1,288 segments
  Cluster  5: 21,880 segments
  Cluster  6: 4,255 segments
  Cluster  7: 243 segments
  Cluster  8: 35,171 segments
  Cluster  9: 3,026 segments
  Cluster 10: 59,360 segments
  Cluster 11: 138,965 segments


## 8 · Assemble and save segments output CSV

Transform the full network into the training-set PCA space and save with cluster labels. This is the lookup table used by NB05 for H3 aggregation.

In [9]:
# Transform full network into training-set PCA space
X_all_scaled = pca_scaler.transform(gdf[PCA_INPUT_COLS].values)
X_all_pca = pca_model.transform(X_all_scaled)

out = pd.DataFrame({
    SEG_ID_COL: gdf[SEG_ID_COL].values,
    "PC1": X_all_pca[:, 0],
    "PC2": X_all_pca[:, 1],
    "PC3": X_all_pca[:, 2],
    "whole_CH_2": cluster_labels
})

out.to_csv(SEGMENTS_PCA_CLUSTERED_CSV, index=False)
print(f"Saved: {SEGMENTS_PCA_CLUSTERED_CSV}")
print(f"Shape: {out.shape}")
print(out.head())

Saved: D:\SID\data\outputs\segments_pca_clustered.csv
Shape: (427100, 5)
   Depthmap_R       PC1       PC2       PC3  whole_CH_2
0           0 -1.665128  0.231207 -0.069443           1
1           1 -1.919675  0.389288  0.428302           1
2           2 -1.714722  0.273306  0.106972           1
3           3 -1.714392  0.265148  0.138261           1
4           4 -1.595786  0.239246  0.503881           1
